# Text Preprocessing

## CountVectorizer vs HashingVectorizer

Vectorizers are used to convert a collection of text documents to a vector representation, thus helping in preprocessing them before applying any model on these text documents. `CountVectorizer` and `HashingVectorizer` both perform the task of vectorizing the text documents. However, there are some differences among them.

One difference is that `HashingVectorizer` does not store the resulting vocabulary (i.e. the unique tokens). Hence, it can be used to learn from data that does not fit into the computer’s main memory. Each mini-batch is vectorized using `HashingVectorizer` so as to guarantee that the input space of the estimator has always the same dimensionality.

With `HashingVectorizer`, each token directly maps to a pre-defined column position in a matrix. For example, if there are 100 columns in the resultant (vectorized) matrix, each token (word) maps to 1 of the 100 columns. The mapping between the word and the position in matrix is done using hashing.

In other words, in `HashingVectorizer`, each token transforms to a column position instead of adding to the vocabulary. Not storing the vocabulary is useful while handling large data sets. This is because holding a huge token vocabulary comprising of millions of words may be a challenge when the memory is limited.

Since `HashingVectorizer` does not store vocabulary, its object not only takes lesser space, it also alleviates any dependence with function calls performed on the previous chunk of data in case of incremental learning.

### Example
Let us take some sample text documents and vectorize them, first using `CountVectorizer` and then `HashingVectorizer`.

In [ ]:
text_documents = [
    'The well-known saying an apple a day keeps the doctor away has a very straightforward, literal meaning, that the eating of fruit maintains good health.',
    'The proverb first appeared in print in 1866 and over 150 years later is advice that we still pass down through generations.',
    'British apples are one of the nations best loved fruit and according to Great British Apples, we consume around 122,000 tonnes of them each year.',
    'But what are the health benefits, and do they really keep the doctor away?'
]

### 1. CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
c_vectorizer = CountVectorizer()

X_c = c_vectorizer.fit_transform(text_documents)
print("Shape of CountVectorizer matrix:", X_c.shape)
print("Vocabulary size:", len(c_vectorizer.vocabulary_))
print(X_c)

### 2. HashingVectorizer

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer

# n_features declares the number of features (columns) in the output feature matrix.
h_vectorizer = HashingVectorizer(n_features=50)
X_h = h_vectorizer.fit_transform(text_documents)

print("Shape of HashingVectorizer matrix:", X_h.shape)
print(X_h[0])

## Combining preprocessing and fitting in Incremental Learning
*(HashingVectorizer along with SGDClassifier)*

We will now use a dataset containing a textual feature that requires preprocessing using a vectorizer. Since we wish to perform incremental learning using `partial_fit()`, we will preprocess (i.e., vectorize) the dataset feature using `HashingVectorizer` and then we will incrementally fit it.

In [ ]:
import pandas as pd
from io import StringIO, BytesIO, TextIOWrapper
from zipfile import ZipFile
import urllib.request

resp = urllib.request.urlopen('https://archive.ics.uci.edu/ml/machine-learning-databases/00331/sentiment%20labelled%20sentences.zip')
zipfile = ZipFile(BytesIO(resp.read()))

data = TextIOWrapper(zipfile.open('sentiment labelled sentences/amazon_cells_labelled.txt'), encoding='utf-8')

df = pd.read_csv(data, sep='\t')
df.columns = ['review', 'sentiment']
df.head()

In [ ]:
print(df.info())
print(df.describe())
print("Unique sentiments:", df.loc[:, 'sentiment'].unique())

In [ ]:
from sklearn.model_selection import train_test_split

X = df.loc[:, 'review']
y = df.loc[:, 'sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
import numpy as np

vectorizer = HashingVectorizer()
classifier = SGDClassifier(penalty='l2', loss='hinge')

# Iteration 1 of partial_fit()
X_train_part1_hashed = vectorizer.fit_transform(X_train[0:400])
y_train_part1 = y_train[0:400]

all_classes = np.unique(df.loc[:, 'sentiment']) 
classifier.partial_fit(X_train_part1_hashed, y_train_part1, classes=all_classes)

X_test_hashed = vectorizer.transform(X_test)
test_score = classifier.score(X_test_hashed, y_test)
print("Test score after Iteration 1:", test_score)

In [ ]:
# Iteration 2 of partial_fit()
X_train_part2_hashed = vectorizer.transform(X_train[400:])
y_train_part2 = y_train[400:]

classifier.partial_fit(X_train_part2_hashed, y_train_part2)
test_score = classifier.score(X_test_hashed, y_test)
print("Test score after Iteration 2:", test_score)

## TF-IDF
- **Term Frequency (TF):** $\text{TF} = \frac{\text{Number of times the term appears in the document}}{\text{Total number of terms in the document}}$
- **Inverse Document Frequency (IDF):** $\text{IDF} = \log \left(\frac{\text{Numbers of documents in the corpus} + 1}{\text{Numbers of documents in the corpus containing the term} + 1}\right)$
- **TF-IDF:** $\text{TF-IDF} = \text{TF} \times \text{IDF}$